# Allscripts SCM Episode Hydration

Derive SCM disease episodes from hydrated `condition_occurrence` records.

## Current definition
Episodes are built for the same **eight chronic-disease ancestor concepts** as the Epic baseline. **`concept_ancestor`** is used so mapped conditions that roll up to those ancestors (typical for text/SNOMED mapping) still create episodes—not only rows whose `condition_concept_id` exactly equals one of the eight IDs.

Text-mapped `condition_concept_id` values are often **non-standard**; the notebook resolves them to a **standard** target with `concept_relationship` (`Maps to`) before the ancestor join, then falls back to the raw id when no map exists.

The eight chronic targets are **SNOMED** ancestors; when the mapped standard concept is not SNOMED (for example ICD10-CM), an extra **`Maps to` hop into SNOMED** is applied so `concept_ancestor` can reach those parents.

- 320128: Essential hypertension
- 432867: Hyperlipidemia
- 201826: Type 2 diabetes mellitus
- 442077: Anxiety disorder
- 140673: Hypothyroidism
- 45768910: Uncomplicated asthma
- 433736: Obesity
- 442588: Obstructive sleep apnea syndrome


In [ ]:
%sql
TRUNCATE TABLE _exponent.omop_scm.episode_event;
TRUNCATE TABLE _exponent.omop_scm.episode;


In [ ]:
%sql
CREATE OR REPLACE TEMP VIEW maps_to_standard AS
SELECT
  cr.concept_id_1,
  MIN(cr.concept_id_2) AS concept_id_2
FROM _exponent.omop.concept_relationship cr
JOIN _exponent.omop.concept c2
  ON c2.concept_id = cr.concept_id_2
  AND c2.invalid_reason IS NULL
  AND c2.standard_concept = 'S'
  AND c2.domain_id = 'Condition'
WHERE cr.relationship_id = 'Maps to'
  AND cr.invalid_reason IS NULL
GROUP BY cr.concept_id_1;

CREATE OR REPLACE TEMP VIEW maps_to_snomed_standard AS
SELECT
  cr.concept_id_1,
  MIN(cr.concept_id_2) AS concept_id_2
FROM _exponent.omop.concept_relationship cr
JOIN _exponent.omop.concept c2
  ON c2.concept_id = cr.concept_id_2
  AND c2.invalid_reason IS NULL
  AND c2.standard_concept = 'S'
  AND c2.domain_id = 'Condition'
  AND c2.vocabulary_id = 'SNOMED'
WHERE cr.relationship_id = 'Maps to'
  AND cr.invalid_reason IS NULL
GROUP BY cr.concept_id_1;

CREATE OR REPLACE TEMP VIEW co_condition_standard AS
SELECT
  co.person_id,
  co.condition_start_date,
  co.condition_concept_id,
  CASE
    WHEN c.standard_concept = 'S' THEN co.condition_concept_id
    ELSE COALESCE(m.concept_id_2, co.condition_concept_id)
  END AS std_condition_concept_id
FROM _exponent.omop_scm.condition_occurrence co
JOIN _exponent.omop.concept c
  ON c.concept_id = co.condition_concept_id
  AND c.invalid_reason IS NULL
LEFT JOIN maps_to_standard m
  ON m.concept_id_1 = co.condition_concept_id
  AND c.standard_concept <> 'S'
WHERE co.condition_concept_id <> 0;

CREATE OR REPLACE TEMP VIEW co_resolved AS
SELECT
  b.person_id,
  b.condition_start_date,
  b.condition_concept_id,
  CASE
    WHEN cv.vocabulary_id = 'SNOMED' THEN b.std_condition_concept_id
    ELSE COALESCE(snm.concept_id_2, b.std_condition_concept_id)
  END AS resolved_concept_id
FROM co_condition_standard b
JOIN _exponent.omop.concept cv
  ON cv.concept_id = b.std_condition_concept_id
  AND cv.invalid_reason IS NULL
LEFT JOIN maps_to_snomed_standard snm
  ON snm.concept_id_1 = b.std_condition_concept_id
  AND cv.vocabulary_id <> 'SNOMED';

CREATE OR REPLACE TEMP VIEW episode_union AS
SELECT
  r.person_id,
  ca.ancestor_concept_id AS episode_object_concept_id,
  r.condition_start_date
FROM co_resolved r
JOIN _exponent.omop.concept_ancestor ca
  ON CAST(ca.descendant_concept_id AS BIGINT) = CAST(r.resolved_concept_id AS BIGINT)
WHERE r.resolved_concept_id IS NOT NULL
  AND ca.ancestor_concept_id IN (
    320128, 432867, 201826, 442077, 140673, 45768910, 433736, 442588
  )

UNION ALL

SELECT
  r.person_id,
  r.resolved_concept_id AS episode_object_concept_id,
  r.condition_start_date
FROM co_resolved r
WHERE r.resolved_concept_id IS NOT NULL
  AND r.resolved_concept_id IN (
    320128, 432867, 201826, 442077, 140673, 45768910, 433736, 442588
  )

UNION ALL

SELECT
  r.person_id,
  ca.ancestor_concept_id AS episode_object_concept_id,
  r.condition_start_date
FROM co_resolved r
JOIN _exponent.omop.concept_ancestor ca
  ON CAST(ca.descendant_concept_id AS BIGINT) = CAST(r.condition_concept_id AS BIGINT)
WHERE r.resolved_concept_id <> r.condition_concept_id
  AND ca.ancestor_concept_id IN (
    320128, 432867, 201826, 442077, 140673, 45768910, 433736, 442588
  );

CREATE OR REPLACE TEMP VIEW episode_candidate AS
SELECT
  u.person_id,
  u.episode_object_concept_id,
  MIN(u.condition_start_date) AS episode_start_date,
  CAST(MIN(u.condition_start_date) AS TIMESTAMP) AS episode_start_datetime,
  CAST(NULL AS DATE) AS episode_end_date,
  CAST(NULL AS TIMESTAMP) AS episode_end_datetime,
  CAST(NULL AS BIGINT) AS episode_parent_id,
  CAST(1 AS INT) AS episode_number,
  CONCAT_WS(
    CHR(31),
    'allscripts_scm',
    'episode',
    CAST(u.episode_object_concept_id AS STRING),
    CAST(u.person_id AS STRING),
    CAST(MIN(u.condition_start_date) AS STRING)
  ) AS episode_source_value,
  'allscripts_scm' AS source_system
FROM episode_union u
GROUP BY u.person_id, u.episode_object_concept_id;


In [ ]:
%sql
-- Diagnostics: if episode_candidate or disease-episode concept is empty, INSERT yields 0 rows.
SELECT
  (SELECT COUNT(*) FROM _exponent.omop_scm.condition_occurrence WHERE condition_concept_id <> 0) AS scm_condition_rows_mapped,
  (SELECT COUNT(*) FROM co_resolved) AS co_resolved_rows,
  (SELECT COUNT(*) FROM _exponent.omop.concept_ancestor) AS concept_ancestor_rows,
  (SELECT COUNT(DISTINCT r.resolved_concept_id) FROM co_resolved r
   INNER JOIN _exponent.omop.concept_ancestor ca
     ON CAST(ca.descendant_concept_id AS BIGINT) = CAST(r.resolved_concept_id AS BIGINT)) AS resolved_concepts_in_ancestor_table,
  (SELECT COUNT(DISTINCT r.resolved_concept_id) FROM co_resolved r
   INNER JOIN _exponent.omop.concept_ancestor ca
     ON CAST(ca.descendant_concept_id AS BIGINT) = CAST(r.resolved_concept_id AS BIGINT)
   WHERE ca.ancestor_concept_id IN (
     320128, 432867, 201826, 442077, 140673, 45768910, 433736, 442588
   )) AS resolved_concepts_under_chronic_ancestors,
  (SELECT COUNT(*) FROM episode_union) AS episode_union_rows,
  (SELECT COUNT(*) FROM episode_candidate) AS episode_candidate_rows,
  (SELECT COUNT(*) FROM _exponent.omop.concept
    WHERE invalid_reason IS NULL
      AND domain_id = 'Episode'
      AND (
        LOWER(TRIM(concept_name)) = 'disease episode'
        OR LOWER(TRIM(concept_name)) = 'disease episodes'
      )) AS disease_episode_concept_matches;


In [ ]:
%sql
INSERT INTO _exponent.omop_scm.episode (
  person_id,
  episode_concept_id,
  episode_start_date,
  episode_start_datetime,
  episode_end_date,
  episode_end_datetime,
  episode_parent_id,
  episode_number,
  episode_object_concept_id,
  episode_type_concept_id,
  episode_source_value,
  episode_source_concept_id
)
SELECT
  ec.person_id,
  de.concept_id AS episode_concept_id,
  ec.episode_start_date,
  ec.episode_start_datetime,
  ec.episode_end_date,
  ec.episode_end_datetime,
  ec.episode_parent_id,
  ec.episode_number,
  ec.episode_object_concept_id,
  32817 AS episode_type_concept_id,
  ec.episode_source_value,
  0 AS episode_source_concept_id
FROM episode_candidate ec
CROSS JOIN (
  SELECT concept_id
  FROM (
    SELECT concept_id, 0 AS pri
    FROM _exponent.omop.concept
    WHERE invalid_reason IS NULL
      AND domain_id = 'Episode'
      AND (
        LOWER(TRIM(concept_name)) = 'disease episode'
        OR LOWER(TRIM(concept_name)) = 'disease episodes'
      )
    UNION ALL
    SELECT MIN(concept_id) AS concept_id, 1 AS pri
    FROM _exponent.omop.concept
    WHERE invalid_reason IS NULL
      AND domain_id = 'Episode'
      AND standard_concept = 'S'
      AND concept_class_id = 'Disease Episode'
    UNION ALL
    SELECT MIN(concept_id) AS concept_id, 2 AS pri
    FROM _exponent.omop.concept
    WHERE invalid_reason IS NULL
      AND domain_id = 'Episode'
      AND concept_class_id = 'Disease Episode'
  ) ranked
  WHERE concept_id IS NOT NULL
  ORDER BY pri, concept_id
  LIMIT 1
) de;


In [ ]:
%sql
SELECT
  e.episode_object_concept_id,
  c.concept_name,
  COUNT(*) AS episode_count
FROM _exponent.omop_scm.episode e
JOIN _exponent.omop.concept c
  ON c.concept_id = e.episode_object_concept_id
GROUP BY e.episode_object_concept_id, c.concept_name
ORDER BY episode_count DESC;